# XVA: The Price After the Price · Episode 15: Bonus: how banks compute XVA

Companion notebook for the video. Every number that appears on screen or in the narration is produced by running this notebook and read from `build/outputs.json`.

> **The market.** All inputs are illustrative, made up for teaching: the AONIA and BBSW curves of Series 1 (`quotes_illustrative.csv`), and CDS spreads, swaption volatilities and XVA parameters (`quotes_xva_illustrative.csv`). Counterparty A, the AA and BB clients and Our bank are fictional.
>
> Educational material only, not financial, accounting or regulatory advice. Capital and margin rules are simplified; consult the current standards.

1. Setup · 2. The running trade · 3. The scale of the problem · 4. Monte Carlo error · 5. Regression: values without formulas · 6. A Bermudan swaption by regression · 7. Sensitivities: bump and revalue · 8. Export

**Running in Google Colab?** Run the next cells first: they install QuantLib (version 1.43, the one used in the video) and write the data files this notebook reads. Then run the rest of the notebook in order.

In [ ]:
# Colab doesn't include QuantLib. Install it (about 30 seconds).
# The video used QuantLib 1.43; drop '==1.43' for the latest.
!pip install QuantLib==1.43

In [ ]:
#@title Data: writes `quotes_illustrative.csv` (run me first) { display-mode: "form" }
# Illustrative quotes, made up for teaching. Not market data.
from pathlib import Path
Path('quotes_illustrative.csv').parent.mkdir(parents=True, exist_ok=True)
Path('quotes_illustrative.csv').write_text("""curve,instrument,tenor,value,unit
AONIA,deposit,O/N,3.85,pct
AONIA,OIS,1M,3.86,pct
AONIA,OIS,3M,3.89,pct
AONIA,OIS,6M,3.93,pct
AONIA,OIS,9M,3.97,pct
AONIA,OIS,1Y,4.00,pct
AONIA,OIS,18M,4.05,pct
AONIA,OIS,2Y,4.08,pct
AONIA,OIS,3Y,4.13,pct
AONIA,OIS,5Y,4.24,pct
AONIA,OIS,7Y,4.35,pct
AONIA,OIS,10Y,4.50,pct
BBSW3M,fixing,3M,4.02,pct
BBSW3M,AONIA/BBSW basis,1Y,13.0,bp
BBSW3M,AONIA/BBSW basis,2Y,14.0,bp
BBSW3M,AONIA/BBSW basis,3Y,15.0,bp
BBSW3M,AONIA/BBSW basis,5Y,15.5,bp
BBSW3M,AONIA/BBSW basis,7Y,16.0,bp
BBSW3M,AONIA/BBSW basis,10Y,16.5,bp
BBSW6M,fixing,6M,4.14,pct
BBSW6M,3s6s basis,1Y,8.0,bp
BBSW6M,3s6s basis,2Y,9.0,bp
BBSW6M,3s6s basis,3Y,10.0,bp
BBSW6M,3s6s basis,5Y,11.0,bp
BBSW6M,3s6s basis,7Y,11.5,bp
BBSW6M,3s6s basis,10Y,12.0,bp
""")
print('wrote quotes_illustrative.csv')

In [ ]:
#@title Data: writes `quotes_xva_illustrative.csv` (run me first) { display-mode: "form" }
# Illustrative quotes, made up for teaching. Not market data.
from pathlib import Path
Path('quotes_xva_illustrative.csv').parent.mkdir(parents=True, exist_ok=True)
Path('quotes_xva_illustrative.csv').write_text("""kind,name,tenor,value,unit
cds,Counterparty A,1Y,60,bp
cds,Counterparty A,2Y,75,bp
cds,Counterparty A,3Y,90,bp
cds,Counterparty A,5Y,120,bp
cds,Counterparty A,7Y,135,bp
cds,Counterparty A,10Y,150,bp
cds,Our bank,1Y,25,bp
cds,Our bank,2Y,30,bp
cds,Our bank,3Y,35,bp
cds,Our bank,5Y,45,bp
cds,Our bank,7Y,55,bp
cds,Our bank,10Y,65,bp
cds,AA client,1Y,20,bp
cds,AA client,2Y,25,bp
cds,AA client,3Y,30,bp
cds,AA client,5Y,40,bp
cds,AA client,7Y,48,bp
cds,AA client,10Y,55,bp
cds,BB client,1Y,180,bp
cds,BB client,2Y,220,bp
cds,BB client,3Y,260,bp
cds,BB client,5Y,320,bp
cds,BB client,7Y,350,bp
cds,BB client,10Y,380,bp
swaption,AUD,1Yx4Y,90,bp normal vol
swaption,AUD,2Yx3Y,90,bp normal vol
swaption,AUD,3Yx2Y,89,bp normal vol
swaption,AUD,4Yx1Y,88,bp normal vol
param,recovery,,0.40,fraction
param,hw_mean_reversion,,0.03,per year
param,funding_spread,,45,bp
param,cost_of_capital,,0.10,fraction per year
param,capital_ratio,,0.08,fraction of risk-weighted assets
param,risk_weight AA client,,0.20,fraction (Basel CRE20.43: AA corporate)
param,risk_weight Counterparty A,,0.75,fraction (Basel CRE20.43: BBB corporate)
param,risk_weight BB client,,1.00,fraction (Basel CRE20.43: BB corporate)
param,cva_risk_weight AA client,,0.03,fraction (Basel MAR50.16: industrials IG)
param,cva_risk_weight Counterparty A,,0.03,fraction (Basel MAR50.16: industrials IG)
param,cva_risk_weight BB client,,0.07,fraction (Basel MAR50.16: industrials HY)
param,fx_vol,,0.10,AUD/USD lognormal volatility
param,cva_risk_weight Counterparty W,,0.12,fraction (Basel MAR50.16: financials HY)
param,risk_weight Counterparty W,,1.00,fraction (Basel CRE20.43: BB corporate)
cds,Counterparty W,1Y,150,bp
cds,Counterparty W,2Y,170,bp
cds,Counterparty W,3Y,190,bp
cds,Counterparty W,5Y,220,bp
cds,Counterparty W,7Y,235,bp
cds,Counterparty W,10Y,250,bp
""")
print('wrote quotes_xva_illustrative.csv')

In [ ]:
# Parameters (papermill overrides these)
output_json = "build/outputs.json"
valuation_date = "2026-09-25"
aud_quotes_file = "quotes_illustrative.csv"
xva_quotes_file = "quotes_xva_illustrative.csv"
n_pairs = 20000   # antithetic pairs: 40,000 simulated paths
seed = 2026

## 1. Setup

In [ ]:
import json
import math
import datetime as dt
from pathlib import Path

import numpy as np
import pandas as pd
import QuantLib as ql

today = ql.DateParser.parseISO(valuation_date)
ql.Settings.instance().evaluationDate = today
cal, dc = ql.Australia(), ql.Actual365Fixed()
out = {"meta": {
    "series": "XVA: The Price After the Price",
    "episode": 15,
    "valuation_date": valuation_date,
    "quantlib_version": ql.__version__,
    "quotes_label": "illustrative (made up for teaching)",
    "generated_at": dt.datetime.now().isoformat(timespec="seconds"),
}}

AONIA and BBSW curves (Series 1, `episodes/_shared/aud_curves.py`):

In [ ]:
# AUD curve family used from Episode 6 on. Copied verbatim into each notebook by make_notebook.py,
# so every notebook runs on its own. Needs: ql, pd, today, cal, dc (defined in the setup cell).

def aonia_index(curve=ql.YieldTermStructureHandle()):
    return ql.OvernightIndex("AONIA", 0, ql.AUDCurrency(), cal, dc, curve)

def bbsw(months, curve=ql.YieldTermStructureHandle()):
    # Set on the first day of each period (no fixing lag), Modified Following, no end-of-month rule.
    return ql.IborIndex(f"BBSW{months}M", ql.Period(months, ql.Months), 0, ql.AUDCurrency(),
                        cal, ql.ModifiedFollowing, False, dc, curve)

def build_curves(quotes):
    """AONIA from OIS quotes; 3M BBSW = AONIA + AONIA/BBSW basis; 6M BBSW = 3M BBSW + 3s6s basis.

    Returns the curves, their handles and the SimpleQuote behind every input, keyed (curve, tenor).
    Changing a quote with setValue() flows through all three curves.
    """
    q = {}
    def handle(row):
        scale = 1e4 if row.unit == "bp" else 100
        q[(row.curve, row.tenor)] = ql.SimpleQuote(row.value / scale)
        return ql.QuoteHandle(q[(row.curve, row.tenor)])

    rows = lambda curve: quotes[quotes.curve == curve].itertuples()
    MF = ql.ModifiedFollowing

    ois_helpers = []
    for r in rows("AONIA"):
        if r.instrument == "deposit":
            h = ql.DepositRateHelper(handle(r), ql.Period(1, ql.Days), 0, cal,
                                     ql.Following, False, dc)
        else:
            h = ql.OISRateHelper(1, ql.Period(r.tenor), handle(r), aonia_index(), paymentLag=2,
                                 paymentFrequency=ql.Annual, paymentCalendar=cal,
                                 convention=MF, endOfMonth=False)
        ois_helpers.append(h)
    aonia = ql.PiecewiseLogLinearDiscount(today, ois_helpers, dc)
    aonia.enableExtrapolation()
    aonia_h = ql.YieldTermStructureHandle(aonia)

    h3 = []
    for r in rows("BBSW3M"):
        if r.instrument == "fixing":
            h3.append(ql.DepositRateHelper(handle(r), bbsw(3)))
        else:  # AONIA + spread vs 3M BBSW, both quarterly
            h3.append(ql.OvernightIborBasisSwapRateHelper(
                handle(r), ql.Period(r.tenor), 1, cal, MF, False,
                aonia_index(aonia_h), bbsw(3), aonia_h))
    bbsw3m = ql.PiecewiseLogLinearDiscount(today, h3, dc)
    bbsw3m.enableExtrapolation()
    bbsw3m_h = ql.YieldTermStructureHandle(bbsw3m)

    h6 = []
    for r in rows("BBSW6M"):
        if r.instrument == "fixing":
            h6.append(ql.DepositRateHelper(handle(r), bbsw(6)))
        else:  # 3M BBSW + spread (quarterly) vs 6M BBSW (semi-annual)
            h6.append(ql.IborIborBasisSwapRateHelper(
                handle(r), ql.Period(r.tenor), 1, cal, MF, False,
                bbsw(3, bbsw3m_h), bbsw(6), aonia_h, False))
    bbsw6m = ql.PiecewiseLogLinearDiscount(today, h6, dc)
    bbsw6m.enableExtrapolation()

    helpers = {"AONIA": ois_helpers, "BBSW3M": h3, "BBSW6M": h6}
    curves = {"AONIA": aonia, "BBSW3M": bbsw3m, "BBSW6M": bbsw6m}
    for c in curves.values():
        c.nodes()  # bootstrap now, in order
    handles = {k: ql.YieldTermStructureHandle(c) for k, c in curves.items()}
    return curves, handles, q, helpers

def vanilla_swap(tenor, fixed_rate, index_months, forecast, discount, notional,
                 receive=True, start=None):
    """AUD vanilla swap: quarterly vs 3M BBSW or semi-annual vs 6M BBSW, ACT/365F, T+1 start."""
    start = start or cal.advance(today, 1, ql.Days)
    end = cal.advance(start, ql.Period(tenor), ql.ModifiedFollowing, False)
    freq = ql.Quarterly if index_months == 3 else ql.Semiannual
    sched = ql.Schedule(start, end, ql.Period(freq), cal, ql.ModifiedFollowing,
                        ql.ModifiedFollowing, ql.DateGeneration.Forward, False)
    side = ql.VanillaSwap.Receiver if receive else ql.VanillaSwap.Payer
    index = bbsw(index_months, forecast)
    sw = ql.VanillaSwap(side, notional, sched, fixed_rate, dc, sched, index, 0.0, dc)
    sw.setPricingEngine(ql.DiscountingSwapEngine(discount))
    return sw

The XVA toolkit used throughout the series (`episodes/_shared/xva.py`):

In [ ]:
# XVA toolkit for Series 4 (XVA: The Price After the Price). Copied verbatim into each notebook, so every
# notebook runs on its own. Needs: ql, np, math, today, cal, dc, and the AUD curve code (aud_curves.py).
#
# Model: one-factor Hull-White on the AONIA curve, r(t) = x(t) + phi(t), dx = -a x dt + sigma dW, fitted to
# today's curve. BBSW forwards = AONIA forwards + a deterministic basis taken from today's curves.

LGD_STANDARD = 0.60  # 40% recovery, the standard assumption for senior unsecured CDS


def credit_curve(spreads_bp, recovery, discount):
    """Bootstrap a flat-hazard credit curve from CDS par spreads {tenor: bp}.

    Standard CDS terms: quarterly premiums on CDS2015 dates, ACT/360, accrual paid on default.
    """
    helpers = [ql.SpreadCdsHelper(s / 1e4, ql.Period(t), 0, ql.WeekendsOnly(), ql.Quarterly, ql.Following,
                                  ql.DateGeneration.CDS2015, ql.Actual360(), recovery, discount)
               for t, s in spreads_bp.items()]
    curve = ql.PiecewiseFlatHazardRate(today, helpers, dc)
    curve.enableExtrapolation()
    curve.nodes()  # bootstrap now, so the helpers are linked and can report implied quotes
    return curve, helpers


def survival(curve, times):
    return np.array([curve.survivalProbability(float(t)) for t in times])


class HullWhite1F:
    """Hand formulas for one-factor Hull-White fitted to a discount curve (Brigo-Mercurio G1++ form)."""

    def __init__(self, curve, a, sigma):
        self.curve, self.a, self.sigma = curve, a, sigma

    def P0(self, t):
        return np.vectorize(lambda u: self.curve.discount(float(u)))(t)

    def B(self, t, T):
        return (1 - np.exp(-self.a * (T - t))) / self.a

    def V(self, t, T):
        a, s, tau = self.a, self.sigma, T - t
        return s**2 / a**2 * (tau + 2 / a * np.exp(-a * tau) - np.exp(-2 * a * tau) / (2 * a) - 1.5 / a)

    def bond(self, t, T, x):
        """Zero-coupon bond price P(t, T) given the state x(t)."""
        return (self.P0(T) / self.P0(t) * np.exp(0.5 * (self.V(t, T) - self.V(0, T) + self.V(0, t))
                                                 - self.B(t, T) * x))

    def forward_measure_x(self, t):
        """Mean and standard deviation of x(t) under the t-forward measure (used to price one exposure date)."""
        a, s = self.a, self.sigma
        mean = -(s**2 / a**2 * (1 - math.exp(-a * t)) - s**2 / (2 * a**2) * (1 - math.exp(-2 * a * t)))
        return mean, s * math.sqrt((1 - math.exp(-2 * a * t)) / (2 * a))

    def simulate(self, grid, n_pairs, seed):
        """Exact joint simulation of x(t) and its integral on the grid (antithetic pairs).

        Returns X (paths x grid) and D, the discount factor exp(-integral of r) from today to each grid time.
        """
        a, s = self.a, self.sigma
        rng = np.random.default_rng(seed)
        z = rng.standard_normal((n_pairs, len(grid) - 1, 2))
        z = np.concatenate([z, -z])
        X = np.zeros((2 * n_pairs, len(grid)))
        I = np.zeros_like(X)
        for k in range(1, len(grid)):
            dt = grid[k] - grid[k - 1]
            e = math.exp(-a * dt)
            bd = (1 - e) / a
            vx = s**2 / (2 * a) * (1 - e * e)
            vi = s**2 / a**2 * (dt - 2 * bd + (1 - e * e) / (2 * a))
            cxi = s**2 / (2 * a**2) * (1 - e)**2
            L = np.linalg.cholesky([[vx, cxi], [cxi, vi]])
            eps = z[:, k - 1] @ L.T
            X[:, k] = X[:, k - 1] * e + eps[:, 0]
            I[:, k] = I[:, k - 1] + X[:, k - 1] * bd + eps[:, 1]
        D = self.P0(grid) * np.exp(-0.5 * self.V(0, grid) - I)
        return X, D


def calibrate_hw(curve_h, a, vol_quotes, index_months):
    """Fix the mean reversion a and calibrate sigma to co-terminal swaptions quoted in normal vol.

    vol_quotes: [(expiry, tenor, normal vol)]. Single-curve helpers (AONIA projects and discounts).
    """
    model = ql.HullWhite(curve_h, a, 0.01)
    fixed_tenor = ql.Period(f"{index_months}M")
    helpers = []
    for expiry, tenor, vol in vol_quotes:
        h = ql.SwaptionHelper(ql.Period(expiry), ql.Period(tenor), ql.QuoteHandle(ql.SimpleQuote(vol)),
                              bbsw(index_months, curve_h), fixed_tenor, dc, dc, curve_h,
                              ql.BlackCalibrationHelper.RelativePriceError, ql.nullDouble(), 1.0, ql.Normal, 0.0)
        h.setPricingEngine(ql.JamshidianSwaptionEngine(model))
        helpers.append(h)
    model.calibrate(helpers, ql.LevenbergMarquardt(), ql.EndCriteria(1000, 100, 1e-10, 1e-10, 1e-10),
                    ql.NoConstraint(), [], [True, False])
    errors_bp = [(h.impliedVolatility(h.modelValue(), 1e-12, 1000, 1e-4, 0.05) - v) * 1e4
                 for h, (_, _, v) in zip(helpers, vol_quotes)]
    return model.params()[1], model, errors_bp


class SwapLegs:
    """A QuantLib vanilla swap reduced to times and amounts the path pricer needs (all times in years from today)."""

    def __init__(self, swap, ois_curve):
        fl = [ql.as_floating_rate_coupon(c) for c in swap.floatingLeg()]
        fx = [ql.as_fixed_rate_coupon(c) for c in swap.fixedLeg()]
        yf = lambda d: dc.yearFraction(today, d)
        self.ts = np.array([yf(c.accrualStartDate()) for c in fl])  # resets (no fixing lag)
        self.te = np.array([yf(c.date()) for c in fl])  # payments
        self.tau = np.array([c.accrualPeriod() for c in fl])
        f_ois = np.array([(ois_curve.discount(c.accrualStartDate()) / ois_curve.discount(c.date()) - 1) / c.accrualPeriod()
                          for c in fl])
        self.basis = np.array([c.rate() for c in fl]) - f_ois  # BBSW minus AONIA forward, held fixed
        self.tp = np.array([yf(c.date()) for c in fx])
        self.ftau = np.array([c.accrualPeriod() for c in fx])
        self.K = fx[0].rate()
        self.N = fl[0].nominal()
        self.sign = 1 if swap.type() == ql.VanillaSwap.Payer else -1  # payer: receive float, pay fixed
        self.end = self.te[-1]

    def value_at(self, hw, t, x, fixings=None):
        """Swap value at time t for states x. Periods that started on or before t use `fixings` (BBSW set on the
        path); without fixings, t must be a reset date or earlier."""
        v = np.zeros_like(np.asarray(x, dtype=float))
        for i in range(len(self.ts)):
            if self.te[i] <= t + 1e-10:
                continue
            p_end = hw.bond(t, self.te[i], x)
            if self.ts[i] < t - 1e-10:
                flt = fixings[:, i] * self.tau[i] * p_end
            else:
                flt = hw.bond(t, self.ts[i], x) - p_end + self.basis[i] * self.tau[i] * p_end
            v = v + flt
        for j in range(len(self.tp)):
            if self.tp[j] > t + 1e-10:
                v = v - self.K * self.ftau[j] * hw.bond(t, self.tp[j], x)
        return self.sign * self.N * v

    def paths(self, hw, grid, X):
        """Swap value on every path at every grid time (each reset must be a grid time)."""
        gi = {round(t, 10): k for k, t in enumerate(grid)}
        fix = np.zeros((X.shape[0], len(self.ts)))
        for i, t in enumerate(self.ts):
            x = X[:, gi[round(t, 10)]]
            fix[:, i] = (1 / hw.bond(t, self.te[i], x) - 1) / self.tau[i] + self.basis[i]
        return np.column_stack([self.value_at(hw, t, X[:, k], fix) for k, t in enumerate(grid)])


def exposure_grid(legs, horizon_years, per_year=12, extra=()):
    """Monthly (by default) exposure dates plus every reset and payment time, so fixings land on the grid."""
    pts = [np.arange(0, horizon_years + 1e-9, 1 / per_year)] + [l.ts for l in legs] + [l.te for l in legs] + [np.asarray(extra)]
    g = np.unique(np.round(np.concatenate(pts), 10))
    return g[g >= 0]


def mean_se(v):
    """Mean and standard error over antithetic paths (pairs are averaged first)."""
    n = len(v) // 2
    pair = 0.5 * (v[:n] + v[n:])
    return float(pair.mean()), float(pair.std(ddof=1) / math.sqrt(n))


def profile(V, D, idx):
    """Exposure profile at grid indices idx: discounted EE and NEE (both positive numbers), undiscounted EE and
    97.5% PFE."""
    pos, neg = np.maximum(V[:, idx], 0), np.maximum(-V[:, idx], 0)
    return {"EE": (D[:, idx] * pos).mean(axis=0), "NEE": (D[:, idx] * neg).mean(axis=0),
            "EE_undisc": pos.mean(axis=0), "PFE": np.quantile(pos, 0.975, axis=0)}


def default_buckets(credit, times):
    """Probability of default in each interval (times[i-1], times[i]], with times[0] = 0."""
    s = survival(credit, times)
    return np.concatenate([[0.0], s[:-1] - s[1:]])


def cva(ee, times, credit, lgd=LGD_STANDARD):
    """Unilateral CVA = LGD x sum of discounted EE x probability of default in the bucket (EE at bucket end)."""
    return lgd * float(np.dot(ee, default_buckets(credit, times)))


def funding_adjustments(ee, nee, times, spread):
    """Simple symmetric FVA: FCA = spread x sum EE dt, FBA = spread x sum NEE dt (discounted exposures)."""
    dt = np.diff(np.concatenate([[0.0], times]))
    return spread * float(np.dot(ee, dt)), spread * float(np.dot(nee, dt))


def sa_ccr_addon(notional, start, end, t, sf=0.005):
    """SA-CCR interest-rate add-on for one unmargined swap seen from time t: SF x notional x supervisory duration."""
    s, e = max(start - t, 0.0), end - t
    if e <= 0:
        return 0.0
    sd = (math.exp(-0.05 * s) - math.exp(-0.05 * e)) / 0.05
    return sf * notional * sd


def schedule_im(notional, remaining):
    """BCBS-IOSCO standardised initial margin for an interest-rate trade: 1% (<2y), 2% (2-5y), 4% (5y+) of notional."""
    if remaining <= 0:
        return 0.0
    return notional * (0.01 if remaining < 2 else 0.02 if remaining <= 5 else 0.04)


# ---- Collateral (Episode 9 on) ----

def lag_index(grid, idx, days):
    """For each grid index in idx, the grid index closest to `days` calendar days earlier (the margin period of risk)."""
    return np.array([int(np.argmin(np.abs(grid - (grid[k] - days / 365)))) for k in idx])


def collateral_held(V_lagged, threshold=0.0):
    """Two-way variation margin with a threshold, as last called: positive = we hold it, negative = we posted it."""
    return np.maximum(V_lagged - threshold, 0) - np.maximum(-V_lagged - threshold, 0)


# ---- Capital (Episode 11 on) ----

def sa_ccr_ead(V, addon, collateral=0.0, alpha=1.4):
    """SA-CCR exposure at default per path: alpha x (RC + multiplier x add-on), RC = max(V - C, 0), with the
    multiplier for negative value or excess collateral, min(1, 0.05 + 0.95 exp((V - C) / (2 x 0.95 x add-on)))."""
    vc = V - collateral
    if addon <= 0:
        return alpha * np.maximum(vc, 0)
    mult = np.minimum(1.0, 0.05 + 0.95 * np.exp(np.minimum(vc, 0) / (2 * 0.95 * addon)))
    return alpha * (np.maximum(vc, 0) + mult * addon)


def ba_cva_capital(ead, maturity, cva_rw, alpha=1.4, ds=0.65):
    """Reduced BA-CVA capital for one counterparty (one netting set): DS x RW / alpha x M x EAD x DF,
    DF = (1 - exp(-0.05 M)) / (0.05 M). With one counterparty, K_reduced equals its stand-alone CVA."""
    if maturity <= 0:
        return 0.0
    df = (1 - math.exp(-0.05 * maturity)) / (0.05 * maturity)
    return ds * cva_rw / alpha * maturity * ead * df


# ---- FX (Episodes 13-14): lognormal AUD/USD on fixed curves ----

def fx_simulate(fwd0, grid, vol, n_pairs, seed):
    """AUD/USD (USD per AUD) on the grid: FX(t) = F(0, t) exp(vol W(t) - vol^2 t / 2), exact, antithetic pairs."""
    rng = np.random.default_rng(seed)
    dW = rng.standard_normal((n_pairs, len(grid) - 1)) * np.sqrt(np.diff(grid))
    dW = np.concatenate([dW, -dW])
    W = np.concatenate([np.zeros((2 * n_pairs, 1)), np.cumsum(dW, axis=1)], axis=1)
    return np.asarray(fwd0) * np.exp(vol * W - 0.5 * vol**2 * grid), W


class FxCashflows:
    """A trade as AUD and USD cash flows seen by us, valued in USD on fixed curves with a simulated FX rate.

    aud: [(time, AUD amount)], usd: [(time, USD amount)], mtm: [(reset time, pay time, AUD notional, factor)]
    for a USD leg whose notional resets to AUD notional x FX(reset): we receive it at the reset and repay it times
    factor (1 + rate x accrual) at the pay time. P_aud and P_usd map a time to today's discount factor.
    """

    def __init__(self, aud, usd, mtm, P_aud, P_usd):
        self.aud, self.usd, self.mtm, self.Pa, self.Pu = aud, usd, mtm, P_aud, P_usd

    def paths(self, grid, FX):
        gi = {round(t, 10): k for k, t in enumerate(grid)}
        V = np.zeros_like(FX)
        for k, t in enumerate(grid):
            a = sum(c * self.Pa(T) for T, c in self.aud if T > t + 1e-10) / self.Pa(t)
            u = sum(c * self.Pu(T) for T, c in self.usd if T > t + 1e-10) / self.Pu(t)
            v = FX[:, k] * a + u
            for ts, tp, n, f in self.mtm:
                if ts <= t + 1e-10 < tp:
                    v = v - n * FX[:, gi[round(ts, 10)]] * f * self.Pu(tp) / self.Pu(t)
            V[:, k] = v
        return V


# ---- Exposure without simulation (Episode 5 on) ----

QUAD_Z = np.linspace(-9, 9, 8001)
QUAD_W = np.exp(-QUAD_Z**2 / 2) / math.sqrt(2 * math.pi) * (QUAD_Z[1] - QUAD_Z[0])


def ee_at_reset(hw, legs_list, t):
    """Discounted EE of a set of swaps at a date where none has a running fixing (e.g. a common reset date):
    P(0,t) x E[max(sum of values, 0)], with x(t) normal under the t-forward measure. Trapezoid rule on a dense
    grid (Gauss-Hermite converges badly on the kink of max)."""
    m, s = hw.forward_measure_x(t)
    x = m + s * QUAD_Z
    v = sum(l.value_at(hw, t, x) for l in legs_list)
    return float(hw.P0(t) * np.dot(QUAD_W, np.maximum(v, 0)))

## 2. The running trade

> **The running trade.** We pay fixed on a 5-year AUD 100 million swap against 6-month BBSW (semi-annual, the AUD convention beyond 3 years) with Counterparty A, a fictional BBB-rated company, with no collateral agreement. The fixed rate is the par rate, so the trade is worth zero today on the risk-free curves.

Credit curves from CDS spreads (Episode 2), a Hull-White model of the AONIA curve calibrated to swaption volatilities (Episode 3), and 40,000 simulated futures for the trade's value (Episode 4).

In [ ]:
curves, handles, q, _ = build_curves(pd.read_csv(aud_quotes_file))
aonia, aonia_h = curves["AONIA"], handles["AONIA"]
xq = pd.read_csv(xva_quotes_file)
PARAM = {r.name: r.value for r in xq[xq.kind == "param"].itertuples()}
CDS = {who: {r.tenor: r.value for r in g.itertuples()} for who, g in xq[xq.kind == "cds"].groupby("name")}
R = PARAM["recovery"]
LGD = 1 - R
credit = {who: credit_curve(spreads, R, aonia_h)[0] for who, spreads in CDS.items()}
VOLS = [(*r.tenor.split("x"), r.value / 1e4) for r in xq[xq.kind == "swaption"].itertuples()]
A_MR = PARAM["hw_mean_reversion"]
SIGMA, hw_ql, calib_err_bp = calibrate_hw(aonia_h, A_MR, VOLS, 6)
hw = HullWhite1F(aonia, A_MR, SIGMA)

NOTIONAL = 100_000_000
par = vanilla_swap("5Y", 0.04, 6, handles["BBSW6M"], aonia_h, NOTIONAL).fairRate()
trade = vanilla_swap("5Y", par, 6, handles["BBSW6M"], aonia_h, NOTIONAL, receive=False)
legs = SwapLegs(trade, aonia)
annuity = abs(trade.fixedLegBPS()) * 1e4 / NOTIONAL   # PV of 1 a year on the fixed schedule, per unit notional
out["trade"] = {"notional": NOTIONAL, "par_pct": par * 100, "years": 5, "npv": trade.NPV(), "annuity": annuity,
                "sigma_bp": SIGMA * 1e4, "mean_reversion": A_MR, "calib_max_err_bp": max(map(abs, calib_err_bp))}
print(f"par {par:.4%}, NPV {trade.NPV():,.2f}, HW sigma {SIGMA * 1e4:.1f}bp (max vol error {max(map(abs, calib_err_bp)):.2f}bp)")

In [ ]:
grid = exposure_grid([legs], legs.end)
X, D = hw.simulate(grid, n_pairs, seed)
V = legs.paths(hw, grid, X)
MON = np.array([k for k, t in enumerate(grid) if abs(t * 12 - round(t * 12)) < 1e-9])   # monthly exposure dates
T_MON = grid[MON]
assert abs(V[0, 0] - trade.NPV()) < 1.0, "path pricer must match QuantLib today"
out["trade"]["paths"] = V.shape[0]
print(f"{V.shape[0]:,} paths x {len(grid)} dates")

## 3. The scale of the problem

Our runs value one trade on 40,000 paths at about 70 dates. A large bank's XVA run: hundreds of thousands of trades,
thousands of paths, a hundred or more dates (illustrative orders of magnitude).

In [ ]:
import time
ours = V.shape[0] * len(grid)
bank = {"trades": 500_000, "paths": 5_000, "dates": 100}
bank_vals = bank["trades"] * bank["paths"] * bank["dates"]
t0 = time.perf_counter()
_ = legs.paths(hw, grid, X)
secs = time.perf_counter() - t0
out["scale"] = {"ours": ours, "bank": bank_vals, "ratio": bank_vals / ours, **bank, "secs": secs,
                "per_val_us": secs / ours * 1e6, "bank_hours_naive": bank_vals * secs / ours / 3600}
out["scale"]

## 4. Monte Carlo error

CVA's standard error shrinks like $1/\sqrt{N}$: four times the paths for half the error.

In [ ]:
cp = credit["Counterparty A"]
dpd = default_buckets(cp, T_MON)
pathwise = LGD * (D[:, MON] * np.maximum(V[:, MON], 0)) @ dpd
rows = []
for n in (625, 2_500, 10_000, 20_000):
    sub = np.concatenate([pathwise[:n], pathwise[n_pairs:n_pairs + n]])
    m_, se = mean_se(sub)
    rows.append({"paths": 2 * n, "cva": m_, "se": se, "se_pct": se / m_ * 100})
out["error"] = {"rows": rows, "se_small": rows[0]["se"], "se_big": rows[-1]["se"], "ratio": rows[0]["se"] / rows[-1]["se"]}
pd.DataFrame(rows)

## 5. Regression: values without formulas

For most trades there's no closed-form future value. American Monte Carlo estimates it by regression: on every path,
take the trade's discounted future cash flows (known once the path is simulated), and regress them on the state at
$t$, here $x(t)$ with a cubic. The fitted curve is the value function. Two years out, compared with the exact
Hull-White value:

In [ ]:
k2 = int(np.argmin(np.abs(grid - 2.0)))
t2 = grid[k2]
# realised discounted cash flows after t2, per path (float coupons fixed on the path, fixed coupons known)
fix = np.zeros((X.shape[0], len(legs.ts)))
for i, ts_ in enumerate(legs.ts):
    kk = int(np.argmin(np.abs(grid - ts_)))
    fix[:, i] = (1 / hw.bond(ts_, legs.te[i], X[:, kk]) - 1) / legs.tau[i] + legs.basis[i]
cf = np.zeros(X.shape[0])
for i in range(len(legs.ts)):
    if legs.te[i] > t2 + 1e-9:
        kk = int(np.argmin(np.abs(grid - legs.te[i])))
        cf += (fix[:, i] - legs.K) * legs.tau[i] * D[:, kk]
cf = legs.sign * NOTIONAL * cf / D[:, k2]               # discounted back to t2
x = X[:, k2]
A_ = np.vstack([np.ones_like(x), x, x**2, x**3]).T
beta = np.linalg.lstsq(A_, cf, rcond=None)[0]
v_reg = A_ @ beta
v_exact = V[:, k2]
ee_reg = float((D[:, k2] * np.maximum(v_reg, 0)).mean())
ee_ex = float((D[:, k2] * np.maximum(v_exact, 0)).mean())
xs = np.linspace(np.quantile(x, 0.01), np.quantile(x, 0.99), 40)
inside = np.where((x >= xs[0]) & (x <= xs[-1]))[0]
pick = np.random.default_rng(7).choice(inside, 400, replace=False)
out["regression"] = {"ee_reg": ee_reg, "ee_exact": ee_ex, "gap_pct": (ee_reg / ee_ex - 1) * 100,
                     "rmse": float(np.sqrt(np.mean((v_reg - v_exact) ** 2))), "cf_sd": float(cf.std()),
                     "scatter_x": (x[pick] * 1e4).tolist(), "scatter_y": (cf[pick] / 1e6).tolist(),
                     "fit_x": (xs * 1e4).tolist(), "fit_y": ((np.vstack([np.ones_like(xs), xs, xs**2, xs**3]).T @ beta) / 1e6).tolist(),
                     "x_range": [float(xs[0] * 1e4), float(xs[-1] * 1e4)],
                     "x_ticks": [{"x": v, "label": f"{v:+d}" if v else "0"} for v in range(-300, 301, 100) if xs[0] * 1e4 <= v <= xs[-1] * 1e4],
                     "exact_y": (legs.value_at(hw, t2, xs, fixings=np.repeat(fix[:1], len(xs), axis=0)) / 1e6).tolist()}
{k: v for k, v in out["regression"].items() if not isinstance(v, list)}

## 6. A Bermudan swaption by regression

The same idea prices early exercise (Longstaff-Schwartz): a payer Bermudan swaption at our fixed rate, exercisable
at each reset from year one into the rest of the swap (single curve). Regression decides, path by path, whether to
exercise. Compared with QuantLib's finite-difference Hull-White engine:

In [ ]:
sc_swap = vanilla_swap("5Y", par, 6, aonia_h, aonia_h, NOTIONAL, receive=False)
sc = SwapLegs(sc_swap, aonia)
sc.basis[:] = 0.0
ex_i = [i for i in range(len(sc.ts)) if sc.ts[i] >= 1.0 - 1e-9][:-1]
cash = np.zeros(X.shape[0])
for i in reversed(ex_i):
    k = int(np.argmin(np.abs(grid - sc.ts[i])))
    exv = sc.value_at(hw, sc.ts[i], X[:, k])
    itm = exv > 0
    if i == ex_i[-1]:
        cash = np.where(itm, D[:, k] * exv, 0.0)
        continue
    xk = X[itm, k]
    B_ = np.vstack([np.ones_like(xk), xk, xk**2, xk**3]).T
    cont = cash[itm] / D[itm, k]
    b = np.linalg.lstsq(B_, cont, rcond=None)[0]
    take = exv[itm] > B_ @ b
    idx = np.where(itm)[0][take]
    cash[idx] = D[idx, k] * exv[idx]
lsm, lsm_se = mean_se(cash)
fl = [ql.as_floating_rate_coupon(c) for c in sc_swap.floatingLeg()]
dates = [fl[j].accrualStartDate() for j in range(ex_i[0], len(fl))] + [fl[-1].date()]
sch = ql.Schedule(dates, cal, ql.Unadjusted)
und = ql.VanillaSwap(ql.VanillaSwap.Payer, NOTIONAL, sch, par, dc, sch, bbsw(6, aonia_h), 0.0, dc)
berm = ql.Swaption(und, ql.BermudanExercise(dates[:-2]))
berm.setPricingEngine(ql.FdHullWhiteSwaptionEngine(hw_ql, 200, 200))
fd = berm.NPV()
euro = ql.Swaption(und, ql.EuropeanExercise(dates[0]))
euro.setPricingEngine(ql.JamshidianSwaptionEngine(hw_ql))
out["bermudan"] = {"lsm": lsm, "se": lsm_se, "fd": fd, "gap_pct": (lsm / fd - 1) * 100, "european": euro.NPV(),
                   "n_ex": len(ex_i), "premium": fd - euro.NPV()}
out["bermudan"]

## 7. Sensitivities: bump and revalue

CVA's sensitivities by bump-and-revalue: one CDS tenor at a time (no re-simulation needed), and one AONIA quote at
a time (each needs a full re-run of the trade's values on the paths, same random numbers).

In [ ]:
EE = profile(V, D, MON)["EE"]
base = cva(EE, T_MON, cp, LGD)
cs01 = []
for tenor in CDS["Counterparty A"]:
    bumped = dict(CDS["Counterparty A"]); bumped[tenor] += 1
    cs01.append({"bucket": tenor, "cs01": cva(EE, T_MON, credit_curve(bumped, R, aonia_h)[0], LGD) - base})
t0 = time.perf_counter()
ir = []
for key in [k for k in q if k[0] == "AONIA"]:
    q[key].setValue(q[key].value() + 0.0001)
    Vb = SwapLegs(trade, aonia).paths(hw, grid, X)
    ir.append({"bucket": key[1], "delta": cva(profile(Vb, D, MON)["EE"], T_MON, cp, LGD) - base})
    q[key].setValue(q[key].value() - 0.0001)
ir_secs = time.perf_counter() - t0
assert abs(trade.NPV()) < 1.0
out["sensitivities"] = {"cs01": cs01, "ir": ir, "n": len(cs01) + len(ir), "ir_secs": ir_secs,
                        "cs01_total": sum(r["cs01"] for r in cs01), "ir_total": sum(r["delta"] for r in ir),
                        "cs01_5y": [r["cs01"] for r in cs01 if r["bucket"] == "5Y"][0]}
{k: v for k, v in out["sensitivities"].items() if not isinstance(v, list)}

## 8. Export for the video

In [ ]:
path = Path(output_json)
path.parent.mkdir(parents=True, exist_ok=True)
path.write_text(json.dumps(out, indent=2, default=float))
print("wrote", path.resolve(), f"({path.stat().st_size / 1024:.0f} KB)")